# Registration Comparison: Manual vs Auto

This notebook compares two registrations on one vessel mesh:
- **manual**: existing `opa_checkpoint.pkl` and `diff_centreline_checkpoint.pkl`
- **auto**: generated with `register_openings_auto_normals()` and auto centerline endpoints

No notebook from the repo is loaded or modified.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import copy
import pickle
import sys

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ensure repo root (folder containing 'ghd') is on sys.path
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ghd").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "ghd").is_dir():
    raise RuntimeError("Could not locate repo root containing 'ghd'. Start Jupyter in /workspace/AneuG or a subfolder.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ghd.fitting.registration import RegistrationwOpeningAlignmentwDifferentiableCentreline

# ----------------------
# User config
# ----------------------
CASE_ROOT = REPO_ROOT / "checkpoints/alignment"
CASE_NAME = "C0029"   # e.g. "USFD_0056"
MESH_FILENAME = "part_aligned.obj"

NUM_OPENINGS = 3
NUM_CEP = 3
AUTO_MIN_LOOP_VERTICES = 24
AUTO_CAST_STEP_SIZE = 2
AUTO_NORMAL_DOT_MIN = 0.72
AUTO_FACE_DOT_MIN = 0.90

SAVE_AUTO_CHECKPOINTS = False
AUTO_OPA_NAME = "opa_checkpoint_auto.pkl"
AUTO_CL_NAME = "diff_centreline_checkpoint_auto.pkl"

case_dir = CASE_ROOT / CASE_NAME
manual_opa_path = case_dir / "opa_checkpoint.pkl"
manual_cl_path = case_dir / "diff_centreline_checkpoint.pkl"

assert (case_dir / MESH_FILENAME).exists(), f"Mesh not found: {case_dir / MESH_FILENAME}"
assert manual_opa_path.exists(), f"Manual opening checkpoint not found: {manual_opa_path}"
assert manual_cl_path.exists(), f"Manual centerline checkpoint not found: {manual_cl_path}"

print(f"Case: {case_dir}")

In [ ]:
def load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def safe_indices(idx, n):
    idx = np.asarray(idx, dtype=np.int64).reshape(-1)
    if idx.size == 0:
        return idx
    return idx[(idx >= 0) & (idx < n)]


def opening_surfaces_from_checkpoint(opa_chk):
    surfaces = []
    rec_v = opa_chk.get("op_rec_v", [])
    rec_f = opa_chk.get("op_rec_f", [])

    if not (isinstance(rec_v, list) and isinstance(rec_f, list)):
        return surfaces
    if len(rec_v) != len(rec_f):
        return surfaces

    for v, f in zip(rec_v, rec_f):
        v = np.asarray(v, dtype=np.float64).reshape(-1, 3)
        f = np.asarray(f, dtype=np.int64).reshape(-1, 3)
        if len(v) < 3 or len(f) < 1:
            continue
        valid = np.all((f >= 0) & (f < len(v)), axis=1)
        f = f[valid]
        if len(f) < 1:
            continue
        surfaces.append((v, f))
    return surfaces


def opening_centers(verts, opa_chk):
    centers = []
    surfaces = opening_surfaces_from_checkpoint(opa_chk)
    if len(surfaces):
        for v, _ in surfaces:
            centers.append(v.mean(axis=0))
        return np.vstack(centers)

    # Fallback: raw picked opening indices
    for idx in opa_chk.get("op_v_indices", []):
        idx = safe_indices(idx, len(verts))
        if idx.size == 0:
            continue
        centers.append(verts[idx].mean(axis=0))
    if len(centers) == 0:
        return np.zeros((0, 3), dtype=np.float64)
    return np.vstack(centers)


def get_cep_points(verts, cl_chk):
    cep_idx = safe_indices(cl_chk.get("diff_cep_registration", []), len(verts))
    return cep_idx, verts[cep_idx] if cep_idx.size else np.zeros((0, 3), dtype=np.float64)


def centerline_polylines_from_paths(verts, paths):
    polylines = []
    if paths is None:
        return polylines
    for path in paths:
        idx = safe_indices(path, len(verts))
        if idx.size >= 2:
            polylines.append(verts[idx])
    return polylines


def centerline_polylines_from_endpoints(reg, endpoint_idx):
    verts = np.asarray(reg.mesh_target.vertices)
    endpoint_idx = safe_indices(endpoint_idx, len(verts)).tolist()
    if len(endpoint_idx) < 2:
        return [], np.asarray(endpoint_idx, dtype=np.int64), None

    center_idx = reg._estimate_bifurcation_index(endpoint_idx)
    endpoint_sorted = reg._sort_endpoint_indices(endpoint_idx, center_idx)
    paths = reg._branch_paths_from_endpoints(endpoint_sorted, center_idx)

    polylines = []
    for path in paths:
        idx = safe_indices(path, len(verts))
        if idx.size >= 2:
            polylines.append(verts[idx])

    return polylines, np.asarray(endpoint_sorted, dtype=np.int64), int(center_idx)


def _add_trace(fig, trace, row=None, col=None):
    if row is None or col is None:
        fig.add_trace(trace)
    else:
        fig.add_trace(trace, row=row, col=col)


def add_mesh(fig, verts, faces, color="lightgray", opacity=0.16, row=None, col=None):
    mesh = go.Mesh3d(
        x=verts[:, 0],
        y=verts[:, 1],
        z=verts[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color=color,
        opacity=opacity,
        name="mesh",
        hoverinfo="skip",
        showlegend=False,
    )
    _add_trace(fig, mesh, row=row, col=col)


def add_opening_surfaces(fig, verts, opa_chk, label, color, row=None, col=None):
    surfaces = opening_surfaces_from_checkpoint(opa_chk)

    if len(surfaces):
        for i, (v, f) in enumerate(surfaces):
            tr = go.Mesh3d(
                x=v[:, 0], y=v[:, 1], z=v[:, 2],
                i=f[:, 0], j=f[:, 1], k=f[:, 2],
                color=color,
                opacity=0.45,
                name=f"{label} opening patch",
                legendgroup=f"{label}_opening_patch",
                hoverinfo="skip",
                showlegend=(i == 0),
            )
            _add_trace(fig, tr, row=row, col=col)
        return

    # Fallback when no reconstructed opening mesh exists in checkpoint
    for i, idx in enumerate(opa_chk.get("op_v_indices", [])):
        idx = safe_indices(idx, len(verts))
        if idx.size == 0:
            continue
        xyz = verts[idx]
        tr = go.Scatter3d(
            x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
            mode="markers",
            marker=dict(size=2, color=color),
            name=f"{label} opening points",
            legendgroup=f"{label}_opening_patch",
            showlegend=(i == 0),
        )
        _add_trace(fig, tr, row=row, col=col)


def add_registration_traces(
    fig,
    verts,
    opa_chk,
    cl_chk,
    centerline_polylines,
    label,
    opening_color,
    centerline_color,
    endpoint_color,
    line_dash="solid",
    bifurcation_idx=None,
    row=None,
    col=None,
):
    # Opening patches
    add_opening_surfaces(fig, verts, opa_chk, label, opening_color, row=row, col=col)

    # Opening centers
    centers = opening_centers(verts, opa_chk)
    if len(centers):
        tr = go.Scatter3d(
            x=centers[:, 0], y=centers[:, 1], z=centers[:, 2],
            mode="markers",
            marker=dict(size=5, color=opening_color),
            name=f"{label} opening centers",
            legendgroup=f"{label}_opening_centers",
            showlegend=True,
        )
        _add_trace(fig, tr, row=row, col=col)

    # Clean centerline (exactly one polyline per endpoint branch)
    for i, xyz in enumerate(centerline_polylines):
        tr = go.Scatter3d(
            x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
            mode="lines",
            line=dict(color=centerline_color, width=7, dash=line_dash),
            name=f"{label} centerline",
            legendgroup=f"{label}_centerline",
            showlegend=(i == 0),
        )
        _add_trace(fig, tr, row=row, col=col)

    # Endpoints
    cep_idx, cep_pts = get_cep_points(verts, cl_chk)
    if len(cep_pts):
        tr = go.Scatter3d(
            x=cep_pts[:, 0], y=cep_pts[:, 1], z=cep_pts[:, 2],
            mode="markers+text",
            marker=dict(size=7, color=endpoint_color, symbol="diamond"),
            text=[str(int(i)) for i in cep_idx],
            textposition="top center",
            name=f"{label} endpoints",
            legendgroup=f"{label}_endpoints",
            showlegend=True,
        )
        _add_trace(fig, tr, row=row, col=col)

    if bifurcation_idx is not None and 0 <= int(bifurcation_idx) < len(verts):
        p = verts[int(bifurcation_idx)]
        tr = go.Scatter3d(
            x=[p[0]], y=[p[1]], z=[p[2]],
            mode="markers",
            marker=dict(size=7, color="black", symbol="x"),
            name=f"{label} bifurcation",
            legendgroup=f"{label}_bif",
            showlegend=True,
        )
        _add_trace(fig, tr, row=row, col=col)


def matched_distances(src_pts, dst_pts):
    if len(src_pts) == 0 or len(dst_pts) == 0:
        return np.array([]), np.array([], dtype=int), np.array([], dtype=int)
    D = np.linalg.norm(src_pts[:, None, :] - dst_pts[None, :, :], axis=-1)
    r, c = linear_sum_assignment(D)
    return D[r, c], r, c



In [ ]:
# Load manual checkpoints
manual_opa = load_pickle(manual_opa_path)
manual_cl = load_pickle(manual_cl_path)

# Build auto registration in memory
args = SimpleNamespace(device="cpu")
reg_auto = RegistrationwOpeningAlignmentwDifferentiableCentreline(
    args=args,
    root=str(CASE_ROOT),
    target=CASE_NAME,
    num_op=NUM_OPENINGS,
    num_cep=NUM_CEP,
    step_size=AUTO_CAST_STEP_SIZE,
)

# 1) Openings first (normal/graph-based)
reg_auto.register_openings_auto_normals(
    min_loop_vertices=AUTO_MIN_LOOP_VERTICES,
    normal_dot_min=AUTO_NORMAL_DOT_MIN,
    face_dot_min=AUTO_FACE_DOT_MIN,
)
# 2) Build opening patch meshes
reg_auto.create_opening_meshes(viz=False)
# 3) Endpoint/Centerline (reuses opening-derived summary if available)
reg_auto.register_centreline_end_points(auto=True)
# 4) Differentiable wave casting
reg_auto._cast_waves(progress=False)

print("Auto registration debug:")
print(reg_auto.auto_registration_debug)

auto_opa = {
    "op_v_indices": copy.deepcopy(reg_auto.op_v_indices),
    "op_v_coords": copy.deepcopy(reg_auto.op_v_coords),
    "op_v_normal": copy.deepcopy(reg_auto.op_v_normal),
    "op_n_mean": copy.deepcopy(reg_auto.op_n_mean),
    "op_rec_v": copy.deepcopy(reg_auto.op_rec_v),
    "op_rec_f": copy.deepcopy(reg_auto.op_rec_f),
    "op_rec_v_indices_map": copy.deepcopy(reg_auto.op_rec_v_indices_map),
    "op_rec_f_map": copy.deepcopy(reg_auto.op_rec_f_map),
    "op_tangent": copy.deepcopy(getattr(reg_auto, "op_tangent", [])),
    "op_cut_points": copy.deepcopy(getattr(reg_auto, "op_cut_points", [])),
}

auto_cl = {
    "diff_cep_registration": copy.deepcopy(reg_auto.cep_registration),
    "wave_loops": copy.deepcopy(reg_auto.wave_loops),
    "centreline_pcd": copy.deepcopy(getattr(reg_auto, "centreline_pcd", None)),
    "centreline_branch_paths": copy.deepcopy(getattr(reg_auto, "centreline_branch_paths", None)),
    "centreline_tangent": copy.deepcopy(getattr(reg_auto, "centreline_tangent", None)),
}

if SAVE_AUTO_CHECKPOINTS:
    with open(case_dir / AUTO_OPA_NAME, "wb") as f:
        pickle.dump(auto_opa, f)
    with open(case_dir / AUTO_CL_NAME, "wb") as f:
        pickle.dump(auto_cl, f)
    print(f"Saved: {case_dir / AUTO_OPA_NAME}")
    print(f"Saved: {case_dir / AUTO_CL_NAME}")

verts = np.asarray(reg_auto.mesh_target.vertices)
faces = np.asarray(reg_auto.mesh_target.triangles)

# Build clean centerline polylines for visualization from endpoint paths (instead of raw wave loops)
manual_centerlines, manual_eps_sorted, manual_bif_idx = centerline_polylines_from_endpoints(
    reg_auto,
    manual_cl.get("diff_cep_registration", []),
)

auto_centerlines = centerline_polylines_from_paths(verts, auto_cl.get("centreline_branch_paths", None))
if len(auto_centerlines) == 0:
    auto_centerlines, auto_eps_sorted, auto_bif_idx = centerline_polylines_from_endpoints(
        reg_auto,
        auto_cl.get("diff_cep_registration", []),
    )
else:
    auto_eps_sorted = safe_indices(auto_cl.get("diff_cep_registration", []), len(verts))
    auto_bif_idx = reg_auto._estimate_bifurcation_index(auto_eps_sorted.tolist()) if len(auto_eps_sorted) >= 2 else None

print("manual openings:", len(manual_opa.get("op_v_indices", [])))
print("auto openings:", len(auto_opa.get("op_v_indices", [])))
print("manual endpoints:", len(np.asarray(manual_cl.get("diff_cep_registration", [])).reshape(-1)))
print("auto endpoints:", len(np.asarray(auto_cl.get("diff_cep_registration", [])).reshape(-1)))
print("manual centerline branches:", len(manual_centerlines), "| points per branch:", [len(p) for p in manual_centerlines])
print("auto centerline branches:", len(auto_centerlines), "| points per branch:", [len(p) for p in auto_centerlines])



In [ ]:
# Overlay view
fig_overlay = go.Figure()
add_mesh(fig_overlay, verts, faces, color="lightgray", opacity=0.14)

add_registration_traces(
    fig_overlay,
    verts,
    manual_opa,
    manual_cl,
    centerline_polylines=manual_centerlines,
    label="manual",
    opening_color="orange",
    centerline_color="crimson",
    endpoint_color="darkred",
    line_dash="dash",
    bifurcation_idx=manual_bif_idx,
)

add_registration_traces(
    fig_overlay,
    verts,
    auto_opa,
    auto_cl,
    centerline_polylines=auto_centerlines,
    label="auto",
    opening_color="deepskyblue",
    centerline_color="cyan",
    endpoint_color="navy",
    line_dash="solid",
    bifurcation_idx=auto_bif_idx,
)

fig_overlay.update_layout(
    title=f"Manual vs Auto Registration Overlay | {CASE_NAME}",
    scene=dict(aspectmode="data"),
    width=1200,
    height=820,
)
fig_overlay.show()

# Side-by-side view
fig_pair = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("manual", "auto"),
)

add_mesh(fig_pair, verts, faces, row=1, col=1)
add_mesh(fig_pair, verts, faces, row=1, col=2)

add_registration_traces(
    fig_pair,
    verts,
    manual_opa,
    manual_cl,
    centerline_polylines=manual_centerlines,
    label="manual",
    opening_color="orange",
    centerline_color="crimson",
    endpoint_color="darkred",
    line_dash="solid",
    bifurcation_idx=manual_bif_idx,
    row=1,
    col=1,
)

add_registration_traces(
    fig_pair,
    verts,
    auto_opa,
    auto_cl,
    centerline_polylines=auto_centerlines,
    label="auto",
    opening_color="deepskyblue",
    centerline_color="cyan",
    endpoint_color="navy",
    line_dash="solid",
    bifurcation_idx=auto_bif_idx,
    row=1,
    col=2,
)

fig_pair.update_layout(
    title=f"Manual vs Auto Registration Side-by-Side | {CASE_NAME}",
    width=1500,
    height=780,
)
fig_pair.update_scenes(aspectmode="data")
fig_pair.show()



In [ ]:
# Numeric comparison
manual_centers = opening_centers(verts, manual_opa)
auto_centers = opening_centers(verts, auto_opa)
op_dist, op_r, op_c = matched_distances(manual_centers, auto_centers)

_, manual_cep = get_cep_points(verts, manual_cl)
_, auto_cep = get_cep_points(verts, auto_cl)
cep_dist, cep_r, cep_c = matched_distances(manual_cep, auto_cep)

manual_tips = np.vstack([p[-1] for p in manual_centerlines]) if len(manual_centerlines) else np.zeros((0, 3))
auto_tips = np.vstack([p[-1] for p in auto_centerlines]) if len(auto_centerlines) else np.zeros((0, 3))
tip_dist, tip_r, tip_c = matched_distances(manual_tips, auto_tips)

rows = []
for i, d in enumerate(op_dist):
    rows.append({
        "type": "opening_center",
        "manual_index": int(op_r[i]),
        "auto_index": int(op_c[i]),
        "distance": float(d),
    })
for i, d in enumerate(cep_dist):
    rows.append({
        "type": "endpoint",
        "manual_index": int(cep_r[i]),
        "auto_index": int(cep_c[i]),
        "distance": float(d),
    })
for i, d in enumerate(tip_dist):
    rows.append({
        "type": "centerline_tip",
        "manual_index": int(tip_r[i]),
        "auto_index": int(tip_c[i]),
        "distance": float(d),
    })

df = pd.DataFrame(rows)
if len(df):
    display(df.sort_values(["type", "manual_index"]).reset_index(drop=True))
else:
    print("No comparable points found.")

if len(op_dist):
    print(f"Opening center distance | mean={op_dist.mean():.6f}, max={op_dist.max():.6f}")
if len(cep_dist):
    print(f"Endpoint distance       | mean={cep_dist.mean():.6f}, max={cep_dist.max():.6f}")
if len(tip_dist):
    print(f"Centerline tip distance | mean={tip_dist.mean():.6f}, max={tip_dist.max():.6f}")

